
# QWEN 2.5 CODER 3B - FINETUNING NOTEBOOK
# Tech Stack Q&A Assistant Training
#
# BEFORE YOU START:
# 1. Make sure you're using a T4 GPU in Colab
#    Go to: Runtime → Change runtime type → T4 GPU
# 2. Upload your 'training_data.jsonl' file to Colab
#    Click the folder icon on left → Upload button
# 3. Run each cell one by one (Shift + Enter)



In [ ]:
# ============================================================================
# STEP 1: INSTALL REQUIRED PACKAGES
# ============================================================================
# This installs Unsloth (optimized training library) and all dependencies
# Takes ~3-5 minutes. You'll see lots of output - that's normal!

%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

print("✅ Installation complete!")

In [ ]:
# ============================================================================
# STEP 2: LOAD THE MODEL
# ============================================================================
# We're loading Qwen2.5-Coder-3B-Instruct in 4-bit quantization
# This makes it fit in the free T4 GPU (15GB VRAM)

from unsloth import FastLanguageModel
import torch

# Configuration
max_seq_length = 2048  # Maximum length of input sequences
dtype = None           # Auto-detect the best dtype for your GPU
load_in_4bit = True    # Use 4-bit quantization (saves memory!)

print("📥 Loading Qwen2.5-Coder-3B-Instruct model...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit",  # Pre-quantized 4-bit model
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...",  # Uncomment if you need a HuggingFace token
)

print("✅ Model loaded successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
📥 Loading Qwen2.5-Coder-3B-Instruct model...
==((====))==  Unsloth 2026.1.4: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Model loaded successfully!


In [ ]:
# ============================================================================
# STEP 3: ADD LORA ADAPTERS
# ============================================================================
# LoRA (Low-Rank Adaptation) allows us to finetune only 1-10% of parameters
# This makes training MUCH faster and uses less memory

print("🔧 Adding LoRA adapters...")

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # LoRA rank - higher = more parameters to train (try 8, 16, 32)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],  # Which layers to apply LoRA
    lora_alpha = 16,      # LoRA scaling factor
    lora_dropout = 0,     # Dropout (0 is optimized for speed)
    bias = "none",        # Bias training (none is optimized)
    use_gradient_checkpointing = "unsloth",  # Saves memory!
    random_state = 3407,  # For reproducibility
    use_rslora = False,   # Rank stabilized LoRA (optional)
    loftq_config = None,  # LoftQ (optional)
)

print("✅ LoRA adapters added!")


🔧 Adding LoRA adapters...


Unsloth 2026.1.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


✅ LoRA adapters added!


In [ ]:
# ============================================================================
# STEP 4: SETUP CHAT TEMPLATE
# ============================================================================
# This tells the model how to format conversations
# Your dataset uses: {"role": "user/assistant", "content": "..."}

from unsloth.chat_templates import get_chat_template

print("💬 Setting up chat template...")

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",  # ChatML format (works great for Qwen)
    # IMPORTANT: This mapping matches YOUR dataset format!
    mapping = {
        "role": "role",           # Your JSON has "role"
        "content": "content",     # Your JSON has "content"
        "user": "user",          # Your JSON has "user"
        "assistant": "assistant" # Your JSON has "assistant"
    },
    map_eos_token = True,  # Maps end token correctly
)

print("✅ Chat template configured!")

Unsloth: Will map <|im_end|> to EOS = <|im_end|>.


💬 Setting up chat template...
✅ Chat template configured!


In [ ]:
# ============================================================================
# STEP 5: LOAD YOUR DATASET
# ============================================================================
# Load your training_data.jsonl file and format it for training

from datasets import load_dataset

print("📂 Loading your dataset...")

# This function converts your data to the format the trainer needs
def formatting_prompts_func(examples):
    """
    Takes your messages and formats them using the chat template
    Input: {"messages": [{"role": "system", ...}, {"role": "user", ...}, ...]}
    Output: {"text": "formatted conversation string"}
    """
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False
        ) for convo in convos
    ]
    return {"text": texts}

# Load your JSONL file
# MAKE SURE: training_data.jsonl is uploaded to Colab!
dataset = load_dataset("json", data_files="training_data.jsonl", split="train")

# Apply the formatting function
dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"✅ Dataset loaded! Total examples: {len(dataset)}")

# Let's preview one example to make sure it looks good
print("\n📝 Preview of first training example:")
print(dataset[0]["text"][:500] + "...")  # First 500 characters

📂 Loading your dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/39 [00:00<?, ? examples/s]

✅ Dataset loaded! Total examples: 39

📝 Preview of first training example:
<|im_start|>system
You are a Next.js, React, and TypeScript expert assistant. You provide clear, accurate, and practical coding help.<|im_end|>
<|im_start|>user
How do I fetch data in a Server Component in Next.js 13+? A: In the App Router, Server Components can directly use async/await to fetch data. You do not need useEffect or SWR for initial data fetching.<|im_end|>
<|im_start|>assistant
TypeScript

// app/page.tsx
async function getData() {
  const res = await fetch('https://api.example.com...


In [ ]:
# ============================================================================
# STEP 6: CONFIGURE TRAINING
# ============================================================================
# Set up the training parameters

from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

print("⚙️ Configuring training parameters...")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,  # Set to True for short sequences (can be 5x faster!)

    args = TrainingArguments(
        # Batch size - how many examples per training step
        per_device_train_batch_size = 2,

        # Gradient accumulation - effectively multiplies batch size
        gradient_accumulation_steps = 4,  # Effective batch size = 2 * 4 = 8

        # Warmup steps - gradually increase learning rate
        warmup_steps = 5,

        # Training duration
        # Option 1: Fixed number of steps
        max_steps = 200,  # 200 steps is good for 70 examples
        # Option 2: Train for full epochs (uncomment below, comment max_steps)
        # num_train_epochs = 1,
        # max_steps = -1,

        # Learning rate - how fast the model learns
        learning_rate = 2e-4,

        # Precision settings (auto-detected based on GPU)
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),

        # Logging - how often to print progress
        logging_steps = 1,

        # Optimizer - AdamW 8-bit saves memory
        optim = "adamw_8bit",

        # Regularization
        weight_decay = 0.01,

        # Learning rate schedule
        lr_scheduler_type = "linear",

        # Random seed for reproducibility
        seed = 3407,

        # Output directory for checkpoints
        output_dir = "outputs",
    ),
)

print("✅ Training configured!")


⚙️ Configuring training parameters...


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/39 [00:00<?, ? examples/s]

✅ Training configured!


In [ ]:
# ============================================================================
# STEP 7: CHECK GPU MEMORY BEFORE TRAINING
# ============================================================================

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"\n🖥️ GPU: {gpu_stats.name}")
print(f"📊 Max memory: {max_memory} GB")
print(f"💾 Memory reserved: {start_gpu_memory} GB")


🖥️ GPU: Tesla T4
📊 Max memory: 14.741 GB
💾 Memory reserved: 2.971 GB


In [ ]:
# ============================================================================
# STEP 8: START TRAINING! 🚀
# ============================================================================
# This is where the magic happens!
# With 70 examples and 200 steps, this should take ~30-40 minutes

print("\n" + "="*70)
print("🚀 STARTING TRAINING!")
print("="*70)
print("\nThis will take approximately 30-40 minutes.")
print("You can monitor progress below. Don't close this tab!\n")

trainer_stats = trainer.train()

print("\n" + "="*70)
print("🎉 TRAINING COMPLETE!")
print("="*70)


The model is already on multiple devices. Skipping the move to device specified in `args`.



🚀 STARTING TRAINING!

This will take approximately 30-40 minutes.
You can monitor progress below. Don't close this tab!



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 39 | Num Epochs = 40 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kethanvr (kethanvr-cmr-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss
1,3.359000
2,3.267800
3,3.259300
4,2.816600
5,3.395300
6,2.814600
7,2.610700
8,2.204400
9,1.953700
10,1.787700


train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇██
train/global_step,▁▁▁▁▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇██
train/grad_norm,█▇█▇▅▃▄▃▄▃▁▄▁▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,███▇▇▇▇▇▇▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
train/loss,█▇▆▄▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
total_flos,3152148862377984.0
train/epoch,40
train/global_step,200
train/grad_norm,0.16307
train/learning_rate,0.0
train/loss,0.036



🎉 TRAINING COMPLETE!


In [ ]:
# ============================================================================
# STEP 10: TEST YOUR MODEL! 🧪
# ============================================================================
# Let's see if your model learned anything!

from unsloth.chat_templates import get_chat_template

print("\n🧪 Testing the finetuned model...\n")

# Re-apply chat template for inference
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
    mapping = {"role": "role", "content": "content", "user": "user", "assistant": "assistant"},
    map_eos_token = True,
)

# Enable fast inference mode
FastLanguageModel.for_inference(model)

# Test question - try one from your training data!
messages = [
    {"role": "user", "content": "How do I use useState in React with TypeScript?"}
]

# Prepare input
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,  # Important for generation!
    return_tensors = "pt",
).to("cuda")

# Generate response
print("🤖 Model's Answer:\n")
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 256,  # Maximum length of response
    use_cache = True,
     pad_token_id = tokenizer.eos_token_id  # Add this line!
)



🧪 Testing the finetuned model...

🤖 Model's Answer:

<|im_start|>user
How do I use useState in React with TypeScript?<|im_end|>
<|im_start|>assistant
TypeScript

const [count, setCount] = useState(0);<|im_end|>


In [ ]:
# ============================================================================
# STEP 11: TRY YOUR OWN QUESTIONS!
# ============================================================================
# Modify the message below to test different questions

print("\n" + "="*70)
print("💡 Try asking your own questions!")
print("="*70)

# CHANGE THIS to test different questions!
test_messages = [
    {"role": "user", "content": "wht is your name"}
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

print("\n🤖 Model's Answer:\n")
text_streamer = TextStreamer(tokenizer)
_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 256,
    use_cache = True,
     pad_token_id = tokenizer.eos_token_id  # Add this line!
)



💡 Try asking your own questions!

🤖 Model's Answer:

<|im_start|>user
wht is your name<|im_end|>
<|im_start|>assistant
My name is Assistant. How can I help you today?<|im_end|>


In [ ]:

# ============================================================================
# STEP 12: SAVE YOUR MODEL
# ============================================================================
# Save the LoRA adapters (lightweight, ~100MB)

print("\n💾 Saving your finetuned model...")

# Local save (in Colab's storage)
model.save_pretrained("my_nextjs_assistant")
tokenizer.save_pretrained("my_nextjs_assistant")

print("✅ Model saved to 'my_nextjs_assistant' folder!")

# Download the model folder to your computer:
# Click the folder icon → right-click 'my_nextjs_assistant' → Download


💾 Saving your finetuned model...
✅ Model saved to 'my_nextjs_assistant' folder!


In [ ]:
#============================================================================
# STEP 13 (OPTIONAL): UPLOAD TO HUGGING FACE
# ============================================================================
# Share your model with the world!

# First, get your token from: https://huggingface.co/settings/tokens
# Then uncomment and run:


HF_TOKEN = ""  # Paste your token here
USERNAME = "Kethanvr"  # Your HuggingFace username
MODEL_NAME = "my_nextjs_assistant"  # Name for your model

model.push_to_hub(
    f"{USERNAME}/{MODEL_NAME}",
    token = HF_TOKEN
)
tokenizer.push_to_hub(
    f"{USERNAME}/{MODEL_NAME}",
    token = HF_TOKEN
)

print(f"✅ Model uploaded to: https://huggingface.co/{USERNAME}/{MODEL_NAME}")



README.md:   0%|          | 0.00/574 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  560kB /  120MB            

Saved model to https://huggingface.co/Kethanvr/my_nextjs_assistant


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ..._assistant/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

✅ Model uploaded to: https://huggingface.co/Kethanvr/my_nextjs_assistant
